
# 02 - Bronze Layer: Data Ingestion
# Extract MLB Data from API and Store Raw JSON


In [0]:

# Import Libraries

import requests
import json
import uuid
import traceback

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from pyspark.sql import Row
from pyspark.sql.types import *

In [0]:

# General Configuration

pipeline_name = "mlb_daily_games_etl"
source_api = "MLB Stats API"
timezone = "America/Santo_Domingo"

pipeline_run_id = str(uuid.uuid4())

current_time_rd = datetime.now(ZoneInfo(timezone))

# We extract yesterday's games because this pipeline runs at 12:00 AM.
execution_date = (current_time_rd - timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Pipeline Name: {pipeline_name}")
print(f"Pipeline Run ID: {pipeline_run_id}")
print(f"Execution Date: {execution_date}")

In [0]:

# Logging Function
# Saves pipeline execution details into the pipeline_logs table


def log_event(layer, status, message, records_processed, started_at):
    ended_at = datetime.now(ZoneInfo(timezone))
    duration_seconds = float((ended_at - started_at).total_seconds())

    log_schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("execution_date", StringType(), True),
        StructField("records_processed", IntegerType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("ended_at", TimestampType(), True),
        StructField("duration_seconds", DoubleType(), True)
    ])

    log_row = [(
        pipeline_name,
        pipeline_run_id,
        layer,
        status,
        message,
        execution_date,
        int(records_processed),
        started_at,
        ended_at,
        duration_seconds
    )]

    log_df = spark.createDataFrame(log_row, schema=log_schema)

    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("mlb_project.pipeline_logs")

In [0]:

# API Extraction Configuration
# Defines the MLB API endpoint and request parameters


bronze_started_at = datetime.now(ZoneInfo(timezone))

base_url = "https://statsapi.mlb.com/api/v1/schedule"

params = {
    "sportId": 1,
    "date": execution_date,
    "hydrate": "team,linescore,probablePitcher,venue"
}

print("API URL:", base_url)
print("Request Parameters:", params)

In [0]:

# Extract Data from MLB API
# Sends a GET request to the MLB Stats API

try:
    response = requests.get(base_url, params=params, timeout=30)
    status_code = int(response.status_code)

    if status_code != 200:
        raise Exception(f"API request failed with status code: {status_code}")

    raw_data = response.json()

    print("API extraction completed successfully.")
    print("Final Request URL:", response.url)

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="bronze",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=bronze_started_at
    )

    raise Exception(error_message)

In [0]:

# Create Bronze DataFrame
# Stores the raw JSON response as a single record


bronze_schema = StructType([
    StructField("execution_date", StringType(), True),
    StructField("source_api", StringType(), True),
    StructField("request_url", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("raw_json", StringType(), True),
    StructField("extracted_at", TimestampType(), True),
    StructField("pipeline_run_id", StringType(), True)
])

bronze_row = [(
    execution_date,
    source_api,
    response.url,
    status_code,
    json.dumps(raw_data),
    datetime.now(ZoneInfo(timezone)),
    pipeline_run_id
)]

bronze_df = spark.createDataFrame(bronze_row, schema=bronze_schema)

display(bronze_df)

In [0]:

# Save Raw Data into Bronze Delta Table
# Appends the raw API response into the Bronze layer


try:
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("mlb_project.bronze_mlb_schedule_raw")

    log_event(
        layer="bronze",
        status="SUCCESS",
        message="Raw API response saved successfully into Bronze table.",
        records_processed=1,
        started_at=bronze_started_at
    )

    print("Bronze data saved successfully.")

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="bronze",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=bronze_started_at
    )

    raise Exception(error_message)

In [0]:

# Bronze Layer Validation
# Shows the latest Bronze records for review


display(
    spark.sql("""
    SELECT
        execution_date,
        source_api,
        status_code,
        extracted_at,
        pipeline_run_id
    FROM mlb_project.bronze_mlb_schedule_raw
    ORDER BY extracted_at DESC
    LIMIT 10
    """)
)